# Unsupervised Domain Adaptation (DANN) for Plant Classification
Goal: Train a classifier that generalizes from single-plant photos to complex quadrat backgrounds using Domain-Adversarial Neural Networks.

In [ ]:
# Papermill parameters
NUM_EPOCHS = None
BATCH_SIZE = None
LEARNING_RATE = None
BACKBONE = None
DOMAIN_LOSS_WEIGHT = None
SOURCE_DATA_PATH = None
TARGET_DATA_PATH = None
GPU_ID = None

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
import mlflow
import numpy as np
from dotenv import load_dotenv
from src.data.datasets import PatchDataset, TestDataset
from sklearn.preprocessing import MultiLabelBinarizer
from torch.amp import autocast
import ast
import csv
import time
import itertools
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
from sklearn.metrics import f1_score

from src.models.dann import DANN
from src.data.datasets import DomainDataset, get_interleaved_loader
from src.utils.metrics import log_dann_metrics, AverageMeter
from src.config.mlflow_init import init_mlflow

# 1. Load environment variables
load_dotenv()

# Initialize MLflow experiment
init_mlflow()

# Override env with papermill params if provided
if NUM_EPOCHS is not None:
    os.environ["NUM_EPOCHS"] = str(NUM_EPOCHS)
if BATCH_SIZE is not None:
    os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
if LEARNING_RATE is not None:
    os.environ["LEARNING_RATE"] = str(LEARNING_RATE)
if BACKBONE is not None:
    os.environ["BACKBONE"] = BACKBONE
if DOMAIN_LOSS_WEIGHT is not None:
    os.environ["DOMAIN_LOSS_WEIGHT"] = str(DOMAIN_LOSS_WEIGHT)
if SOURCE_DATA_PATH is not None:
    os.environ["SOURCE_DATA_PATH"] = SOURCE_DATA_PATH
if TARGET_DATA_PATH is not None:
    os.environ["TARGET_DATA_PATH"] = TARGET_DATA_PATH
if GPU_ID is not None:
    os.environ["GPU_ID"] = str(GPU_ID)

REQUIRED_ENV_VARS = [
    "SOURCE_DATA_PATH",
    "TARGET_DATA_PATH",
    "BACKBONE",
    "LEARNING_RATE",
    "DOMAIN_LOSS_WEIGHT",
]


def validate_env():
    missing = [var for var in REQUIRED_ENV_VARS if os.getenv(var) is None]
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {', '.join(missing)}"
        )
    print("Environment variables validated successfully.")


validate_env()

GPU_ID = os.getenv("GPU_ID", "0")
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Data Preparation

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((518, 518)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Load metadata (Paths defined in .env)
source_path = os.getenv("SOURCE_DATA_PATH")
target_path = os.getenv("TARGET_DATA_PATH")
data_dir = os.getenv("DATA_DIR", "data")

train_metadata_path = os.path.join(data_dir, "train_metadata_split.csv")
val_metadata_path = os.path.join(data_dir, "val_metadata_split.csv")

if os.path.exists(train_metadata_path):
    source_df = pd.read_csv(train_metadata_path, sep=";")
    # Ensure we have an 'image_path' column for DomainDataset
    if "image_path" not in source_df.columns:
        possible_cols = ["image_name", "image_id", "id"]
        for col in possible_cols:
            if col in source_df.columns:
                source_df["image_path"] = source_df[col]
                break
    print(f"Loaded {len(source_df)} source training samples.")
else:
    print(f"Warning: {train_metadata_path} not found. Using empty placeholder.")
    source_df = pd.DataFrame({"image_path": [], "species_id": []})

if os.path.exists(val_metadata_path):
    target_val_df = pd.read_csv(val_metadata_path, sep=";")
    if "image_path" not in target_val_df.columns:
        possible_cols = ["image_name", "image_id", "id"]
        for col in possible_cols:
            if col in target_val_df.columns:
                target_val_df["image_path"] = target_val_df[col]
                break
    print(f"Loaded {len(target_val_df)} source validation samples.")
else:
    print(f"Warning: {val_metadata_path} not found. Using empty placeholder.")
    target_val_df = pd.DataFrame({"image_path": [], "species_id": []})

if os.path.exists(target_path):
    import glob

    # Recursively find all images in the target path
    target_pattern = os.path.join(target_path, "**", "*.[jJ][pP][gG]")
    target_paths = glob.glob(target_pattern, recursive=True)
    # Keep only the relative path for the dataframe
    target_images = [os.path.relpath(p, target_path) for p in target_paths]
    target_df = pd.DataFrame({"image_path": target_images})
    print(f"Loaded {len(target_df)} target samples (unlabeled).")
else:
    print(f"Warning: target path {target_path} not found. Using empty placeholder.")
    target_df = pd.DataFrame({"image_path": []})

if len(source_df) == 0:
    raise ValueError(
        "Source dataset is empty. Ensure 'data/train_metadata_split.csv' exists and SOURCE_DATA_PATH is correct."
    )

source_ds = DomainDataset(source_path, source_df, domain_label=0, transform=transform)
target_ds = DomainDataset(
    target_path,
    target_df,
    domain_label=1,
    transform=transform,
    is_source=False,
)
target_val_ds = DomainDataset(
    source_path, target_val_df, domain_label=0, transform=transform, is_source=True
)

batch_size = int(os.getenv("BATCH_SIZE", 32))
source_loader = DataLoader(source_ds, batch_size=batch_size, shuffle=True)
if len(target_df) == 0:
    raise ValueError(
        f"Target dataset is empty. Check if TARGET_DATA_PATH ({target_path}) contains images."
    )

target_loader = DataLoader(target_ds, batch_size=batch_size, shuffle=True)
target_val_loader = (
    DataLoader(target_val_ds, batch_size=batch_size, shuffle=False)
    if len(target_val_df) > 0
    else None
)

## 2. Model Initialization

In [ ]:
backbone = os.getenv("BACKBONE")
# Set num_classes dynamically based on metadata
num_classes = int(source_df["species_id"].max() + 1)
print(f"Initializing DANN with {num_classes} classes.")
model = DANN(backbone, num_classes).to(DEVICE)

lr = float(os.getenv("LEARNING_RATE"))
domain_loss_weight = float(os.getenv("DOMAIN_LOSS_WEIGHT", 1.0))

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion_class = nn.CrossEntropyLoss()
# Use BCEWithLogitsLoss for better numerical stability
criterion_domain = nn.BCEWithLogitsLoss()

## 3. Training Loop with Adversarial Step

In [ ]:
def train_epoch(
    model, source_loader, target_loader, optimizer, epoch, num_epochs, domain_weight
):
    import torch

    torch.cuda.empty_cache()
    model.train()
    metrics = {
        "loss_class": AverageMeter(),
        "loss_domain_s": AverageMeter(),
        "loss_domain_t": AverageMeter(),
    }

    n_batches = len(source_loader)
    if n_batches == 0:
        return {k: 0.0 for k in metrics.keys()}

    for i, (source_batch, target_batch) in enumerate(
        zip(source_loader, itertools.cycle(target_loader))
    ):
        # 1. Schedule alpha (Gradient Reversal Scaling)
        p = float(i + epoch * n_batches) / (num_epochs * n_batches)
        alpha = 2.0 / (1.0 + np.exp(-10 * p)) - 1

        # 2. Source domain pass
        s_img, s_cls_label, s_dom_label = source_batch
        s_img, s_cls_label = s_img.to(DEVICE), s_cls_label.to(DEVICE)
        s_dom_label = s_dom_label.float().to(DEVICE).view(-1, 1)

        optimizer.zero_grad()
        s_cls_out, s_dom_out = model(s_img, alpha)

        loss_s_cls = criterion_class(s_cls_out, s_cls_label)
        loss_s_dom = criterion_domain(s_dom_out, s_dom_label)

        # 3. Target domain pass
        t_img, _, t_dom_label = target_batch
        t_img = t_img.to(DEVICE)
        t_dom_label = t_dom_label.float().to(DEVICE).view(-1, 1)

        _, t_dom_out = model(t_img, alpha)
        loss_t_dom = criterion_domain(t_dom_out, t_dom_label)

        # 4. Total loss and update (apply domain_loss_weight)
        loss = loss_s_cls + domain_weight * (loss_s_dom + loss_t_dom)
        loss.backward()
        optimizer.step()

        metrics["loss_class"].update(loss_s_cls.item())
        metrics["loss_domain_s"].update(loss_s_dom.item())
        metrics["loss_domain_t"].update(loss_t_dom.item())

    return {k: v.avg for k, v in metrics.items()}

## 4. Validation & Visualization

In [ ]:
def validate_domain_invariance(model, source_loader, target_loader, num_samples=500):
    """Extract features and visualize domain overlap using t-SNE."""
    model.eval()
    features_list = []
    domains_list = []

    with torch.no_grad():
        # Sample from source
        count = 0
        for imgs, _, doms in source_loader:
            feats = model.feature_extractor(imgs.to(DEVICE))
            features_list.append(feats.cpu().numpy())
            domains_list.append(doms.numpy())
            count += imgs.size(0)
            if count >= num_samples:
                break

        # Sample from target
        count = 0
        for imgs, _, doms in target_loader:
            feats = model.feature_extractor(imgs.to(DEVICE))
            features_list.append(feats.cpu().numpy())
            domains_list.append(doms.numpy())
            count += imgs.size(0)
            if count >= num_samples:
                break

    if not features_list:
        print("No data available for visualization.")
        return

    features = np.concatenate(features_list, axis=0)
    domains = np.concatenate(domains_list, axis=0)

    # t-SNE projection
    tsne = TSNE(n_components=2, random_state=42)
    feats_2d = tsne.fit_transform(features)

    # Plot
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        x=feats_2d[:, 0], y=feats_2d[:, 1], hue=domains, palette="viridis", alpha=0.6
    )
    plt.title("DANN Feature Space: Source (0) vs Target (1)")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.show()

    # Log plot to MLflow
    plt.savefig("tsne_overlap.png")
    mlflow.log_artifact("tsne_overlap.png")

In [ ]:
def evaluate_target(model, target_val_loader):
    """Evaluate species classification performance on target domain (quadrats)."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels, _ in target_val_loader:
            imgs = imgs.to(DEVICE)
            cls_out, _ = model(imgs)
            preds = torch.argmax(cls_out, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())

    if not all_preds:
        return {"target/accuracy": 0.0, "target/macro_f1": 0.0}

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = np.mean(all_preds == all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return {"target/accuracy": acc, "target/macro_f1": f1}

## 5. Main Execution

In [ ]:
num_epochs = int(os.getenv("NUM_EPOCHS", 20))
with mlflow.start_run() as run:
    mlflow.log_params(
        {
            "backbone": backbone,
            "learning_rate": lr,
            "batch_size": batch_size,
            "domain_loss_weight": domain_loss_weight,
        }
    )

    for epoch in range(num_epochs):
        epoch_metrics = train_epoch(
            model,
            source_loader,
            target_loader,
            optimizer,
            epoch,
            num_epochs,
            domain_loss_weight,
        )

        # Validate and Log
        target_metrics = evaluate_target(model, target_val_loader)
        combined_metrics = {**epoch_metrics, **target_metrics}

        log_dann_metrics(combined_metrics, step=epoch)
        print(f"Epoch {epoch}: {combined_metrics}")

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            validate_domain_invariance(model, source_loader, target_loader)
    # Log final model to MLflow
    print("Logging model to MLflow...")
    mlflow.pytorch.log_model(model, "model")

In [ ]:
## 6. Submission Generation & Final Evaluation

print("Starting inference for submission.csv...")
model.eval()

# Inference Hyperparameters
inf_batch_size = 64
min_score = 0.1
top_k_tile = 2
patch_size = 518
stride = 259

image_to_tensor = transforms.ToTensor()
# Note: matching baseline directory structure
test_dataset = TestDataset(
    image_folder=os.path.join(
        data_dir, "PlantCLEF2025_test_images", "PlantCLEF2025_test_images"
    ),
    patch_size=patch_size,
    stride=stride,
    use_pad=True,
    transform=image_to_tensor,
)
test_loader = DataLoader(test_dataset, batch_size=1, num_workers=4, pin_memory=True)

image_predictions = {}
batch_time = AverageMeter()
end = time.time()

with torch.no_grad():
    for batch_idx, (patches, image_path) in enumerate(test_loader):
        image_results = {}
        quadrat_id = os.path.splitext(os.path.basename(image_path[0]))[0]

        # Use model's expected normalization (matching training)
        transform_patch = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        patch_ds = PatchDataset(patches[0], transform=transform_patch)
        patch_loader = DataLoader(patch_ds, batch_size=inf_batch_size, shuffle=False)

        for batch_patches in patch_loader:
            batch_patches = batch_patches.to(DEVICE)

            with autocast(DEVICE.type):
                # DANN forward returns (class_output, domain_output)
                outputs, _ = model(batch_patches)
                probabilities = torch.nn.functional.softmax(outputs, dim=1)

                top_probs, top_indices = torch.topk(probabilities, top_k_tile)
                top_probs = top_probs.cpu().numpy()
                top_indices = top_indices.cpu().numpy()

                for top_tile_indices, top_tile_probs in zip(top_indices, top_probs):
                    for top_idx, top_prob in zip(top_tile_indices, top_tile_probs):
                        species_id = int(top_idx)
                        if top_prob > min_score:
                            if (
                                species_id not in image_results
                                or image_results[species_id] < top_prob
                            ):
                                image_results[species_id] = top_prob

        image_predictions[quadrat_id] = list(image_results.keys())
        batch_time.update(time.time() - end)
        end = time.time()

        if batch_idx % 20 == 0:
            print(
                f"Inference: [{batch_idx}/{len(test_loader)}] Time {batch_time.val:.3f}"
            )

# Save predictions
submission_path = "submission.csv"
df_sub = pd.DataFrame(
    list(image_predictions.items()), columns=["quadrat_id", "species_ids"]
)
df_sub["species_ids"] = df_sub["species_ids"].apply(str)
df_sub.to_csv(submission_path, sep=",", index=False, quoting=csv.QUOTE_ALL)

mlflow.log_artifact(submission_path)
print(f"Submission saved to {submission_path} and logged to MLflow.")

# Optional Evaluation logic from baseline.ipynb
ground_truth_path = os.path.join(data_dir, "validation_ground_truth.csv")
if os.path.exists(ground_truth_path):
    print("Ground truth found. Computing final metrics...")
    df_gt = pd.read_csv(ground_truth_path)

    def _parse_species(x):
        if isinstance(x, str):
            x = ast.literal_eval(x)
        return x if isinstance(x, list) else [x]

    df_gt["species_ids"] = df_gt["species_ids"].apply(_parse_species)
    df_sub["predicted_species_ids"] = df_sub["quadrat_id"].map(image_predictions)
    df_eval = pd.merge(
        df_gt,
        df_sub[["quadrat_id", "predicted_species_ids"]],
        on="quadrat_id",
        how="inner",
    )

    if not df_eval.empty:
        from sklearn.metrics import f1_score

        mlb = MultiLabelBinarizer()
        mlb.fit(pd.concat([df_eval["species_ids"], df_eval["predicted_species_ids"]]))
        y_true = mlb.transform(df_eval["species_ids"])
        y_pred = mlb.transform(df_eval["predicted_species_ids"])

        f1_samples = f1_score(y_true, y_pred, average="samples", zero_division=0)
        mlflow.log_metric("final_val_f1_samples", f1_samples)
        print(f"Final Validation F1 (Samples): {f1_samples:.4f}")